# İstanbul İBB Wi-Fi Yeni Kullanıcı Verisi

**Önerme:** Yabancı abonelerin oranı yaz aylarında (Haziran–Ağustos) kış aylarından (Aralık–Şubat) yüksektir.


**Veri seti:** İBB Açık Veri Portalı, *Günlük İBB Wi-Fi Yeni Kullanıcı Verisi*


## Adım 1: Kütüphaneleri yükle ve veriyi oku

In [ ]:
import pandas as pd
import numpy as np

# Tarihleri okuma sırasında tarih tipine dönüştür
df = pd.read_csv("Veri_Seti.csv", parse_dates=["SUBSCRIPTION_DATE"])

print("Satır ve sütun sayısı:", df.shape)
df.head()

: 

## Adım 2:  Veri yapısı ve kalite kontrolleri
Eksik değerler, tekrarlar ve analizde kullanılacak kategoriler kontrol edilir.

In [ ]:
# Sütun adları, veri tipleri ve dolu/boş sayıları
df.info()

In [ ]:
print("Tarih aralığı:", df["SUBSCRIPTION_DATE"].min().date(), "–", df["SUBSCRIPTION_DATE"].max().date())
print("\nEksik değer sayıları:")
print(df.isna().sum())
print("\nTamamen tekrar eden satır sayısı:", df.duplicated().sum())
print("\nAbone türü kayıt sayıları:")
print(df["SUBSCRIBER_DOMESTIC_FOREIGN"].value_counts(dropna=False))
print("\nAbone sayısı özeti:")
df["NUMBER_OF_SUBSCRIBER"].describe()

In [ ]:
# Koordinatlar bu analizde kullanılmıyor; yalnızca İstanbul dışı olabilecek boylamları sayıyoruz.
hatali_koordinat = (df["LONGITUDE"] < 27) | (df["LONGITUDE"] > 30)
print("Boylamı 27–30 aralığı dışında kalan satır sayısı:", int(hatali_koordinat.sum()))

## Adım 3: Analiz verisini hazırlama
“Bilinmiyor” kategorisi yerli/yabancı oranı hesabına dahil edilmez. Aralık ayı, izleyen yılın kış sezonuna atanır; böylece örneğin Aralık 2019–Şubat 2020 dönemi 2020 kışı olur.


In [ ]:
analiz = df.loc[
    df["SUBSCRIBER_DOMESTIC_FOREIGN"].isin(["Yerli", "Yabancı"]),
    ["SUBSCRIPTION_DATE", "SUBSCRIBER_DOMESTIC_FOREIGN", "NUMBER_OF_SUBSCRIBER"]
].copy()

analiz["ay"] = analiz["SUBSCRIPTION_DATE"].dt.month
analiz["yil"] = analiz["SUBSCRIPTION_DATE"].dt.year
analiz["mevsim"] = analiz["ay"].map({12: "Kış", 1: "Kış", 2: "Kış", 6: "Yaz", 7: "Yaz", 8: "Yaz"})
analiz["mevsim_yili"] = np.where(analiz["ay"].eq(12), analiz["yil"] + 1, analiz["yil"])

mevsim_verisi = analiz.dropna(subset=["mevsim"]).copy()
print("Yerli/yabancı olarak bilinen kayıt sayısı:", len(analiz))
print("Mevsim karşılaştırmasına giren satır sayısı:", len(mevsim_verisi))
print("Mevsim dışı aylarda kalan satır sayısı:", len(analiz) - len(mevsim_verisi))

## Adım 4: Mevsim toplamları ve yabancı abone oranı
Oran, yabancı abone sayısının yerli ve yabancı abone toplamına bölünmesiyle hesaplanır. NUMBER_OF_SUBSCRIBER değerleri toplandığı için sonuçlar satır oranı değil, abone sayısı ağırlıklı orandır.

In [ ]:
mevsim_toplam = mevsim_verisi.pivot_table(
    index="mevsim",
    columns="SUBSCRIBER_DOMESTIC_FOREIGN",
    values="NUMBER_OF_SUBSCRIBER",
    aggfunc="sum",
    fill_value=0
)
mevsim_toplam["Toplam abone"] = mevsim_toplam["Yerli"] + mevsim_toplam["Yabancı"]
mevsim_toplam["Yabancı oranı (%)"] = mevsim_toplam["Yabancı"] / mevsim_toplam["Toplam abone"] * 100
mevsim_toplam.round(1)

## Adım 5: Yıl bazında oran ve veri kapsamı
Yıl bazında tablo, toplu oranın hangi yıllardan ve ne ölçüde gözlemden etkilendiğini gösterir. Tarih sayısı, o sezon için CSV’de en az bir kayıt bulunan farklı günlerin sayısıdır; tam veri kapsamını tek başına garanti etmez.

In [ ]:
yil_mevsim = mevsim_verisi.pivot_table(
    index="mevsim_yili",
    columns=["mevsim", "SUBSCRIBER_DOMESTIC_FOREIGN"],
    values="NUMBER_OF_SUBSCRIBER",
    aggfunc="sum",
    fill_value=0
)

# Eksik kategori sütunları varsa sıfırla
yil_mevsim = yil_mevsim.reindex(
    columns=pd.MultiIndex.from_product([["Kış", "Yaz"], ["Yabancı", "Yerli"]]),
    fill_value=0
)

for mevsim in ["Kış", "Yaz"]:
    toplam = yil_mevsim[(mevsim, "Yabancı")] + yil_mevsim[(mevsim, "Yerli")]
    yil_mevsim[(mevsim, "Yabancı oranı (%)")] = np.where(
        toplam > 0,
        yil_mevsim[(mevsim, "Yabancı")] / toplam * 100,
        np.nan
    )

kapsam = mevsim_verisi.groupby(["mevsim_yili", "mevsim"]).agg(
    kayit_satiri=("NUMBER_OF_SUBSCRIBER", "size"),
    veri_bulunan_gun=("SUBSCRIPTION_DATE", "nunique")
)

oran_sutunlari = [(m, "Yabancı oranı (%)") for m in ["Kış", "Yaz"]]
yil_oranlari = yil_mevsim[oran_sutunlari].copy()
yil_oranlari.columns = ["Kış oranı (%)", "Yaz oranı (%)"]

# Kapsam bilgisini satır-sütun yapısına çevirip oranlarla birleştir
kapsam_tablo = kapsam.unstack("mevsim")
kapsam_tablo.columns = [f"{mevsim} {olcu}" for olcu, mevsim in kapsam_tablo.columns]
yil_ozeti = yil_oranlari.join(kapsam_tablo)
yil_ozeti.round(1)

## Adım 6: Sonuç
Önermenin yönü, yaz ve kış oranlarının karşılaştırılmasıyla değerlendirilir. Toplu oran veri hacmine göre ağırlıklıdır; yıl bazındaki oranlar ise her yılın görünümünü ayrı ayrı verir. Bu çalışma betimleyicidir; tek başına istatistiksel anlamlılık veya nedensellik göstermez.

In [ ]:
kis_orani = mevsim_toplam.loc["Kış", "Yabancı oranı (%)"]
yaz_orani = mevsim_toplam.loc["Yaz", "Yabancı oranı (%)"]
fark = yaz_orani - kis_orani

print(f"Toplu kış yabancı oranı: %{kis_orani:.1f}")
print(f"Toplu yaz yabancı oranı: %{yaz_orani:.1f}")
print(f"Fark (yaz - kış): {fark:.1f} yüzde puan")

if yaz_orani > kis_orani:
    print("Toplu veriye göre önerme destekleniyor.")
else:
    print("Toplu veriye göre önerme desteklenmiyor.")

gecerli_yillar = yil_ozeti.dropna(subset=["Kış oranı (%)", "Yaz oranı (%)"])
if not gecerli_yillar.empty:
    fark_yil = gecerli_yillar["Yaz oranı (%)"] - gecerli_yillar["Kış oranı (%)"]
    print(f"\nHer iki mevsimde veri bulunan sezon yılı sayısı: {len(gecerli_yillar)}")
    print(f"Bu yıllardaki yaz-kış farklarının ortalaması: {fark_yil.mean():.1f} yüzde puan")
    print(f"Yaz oranının kış oranından yüksek olduğu yıl sayısı: {int((fark_yil > 0).sum())} / {len(fark_yil)}")

## SONUÇ/RAPOR
- Toplu hesaplamada yabancı abone oranı yazın %17,0, kışın %22,0 bulunmuştur.
- Yaz-kış farkı −5,1 yüzde puandır; bu nedenle önerme desteklenmemektedir.
- Yıl bazında yaz oranı yalnızca 6 yılın 1’inde kış oranını aşmıştır. 
- Ancak mevsimlerin veri kapsamı eşit olmadığından sonuçlar temkinli yorumlanmalıdır. 